In [ ]:
from pyspark.sql import SparkSession, DataFrame, Row
from pyspark.sql.functions import col, to_timestamp, unix_timestamp, when, lit, current_timestamp, coalesce, regexp_replace, length, lower
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, DateType
from pyspark.sql.functions import regexp_replace, trim, concat_ws, lit, to_date, least, greatest, when, current_date, udf

In [ ]:

# Initialize Spark session with legacy time parser
spark = SparkSession.builder \
    .appName("VesselsBronzeToSilverTransformation") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .config("spark.eventLog.logBlockUpdates.enabled", True)\
    .enableHiveSupport() \
    .getOrCreate()

In [ ]:
vessel_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("url", StringType(), True),
    StructField("type", StringType(), True),
    StructField("year_built", IntegerType(), True),
    StructField("gross_tonnage", IntegerType(), True),
    StructField("deadweight", IntegerType(), True),
    StructField("length_m", DoubleType(), True),
    StructField("beam_m", DoubleType(), True),
    StructField("detail_link", StringType(), True),
    StructField("departure_date", StringType(), True),
    StructField("last_port_country", StringType(), True),
    StructField("last_port_name", StringType(), True),
    StructField("arrival_date", StringType(), True),
    StructField("destination_port_country", StringType(), True),
    StructField("destination_port_name", StringType(), True),
    StructField("destination_port_lat", StringType(), True),
    StructField("destination_port_lon", StringType(), True),
    StructField("reported_status", StringType(), True),
    StructField("report_date", DateType(), True),
    StructField("ingestion_date", DateType(), True)
])

vessels_path = "hdfs://localhost:9000/home/itversity/bronze/vessels"

In [ ]:
df = spark.read.option("header", "true").schema(vessel_schema).csv(vessels_path)
# df = df.filter((df["destination_port_lon"].isNotNull()) & ( df["destination_port_lon"] != "-") )
cond = (df["last_port_name"].isNotNull() & df["destination_port_name"].isNotNull())
df = df.filter(cond)
# df.show()

In [ ]:
cond = (df["destination_port_lon"].isNotNull()) & (df["destination_port_lon"] != "-") 
# cond = (df["destination_port_lon"] == df["destination_port_lon"])
non_null_count = df.filter(cond).count()
print(f"Number of non-null values in column Y: {non_null_count}")

In [6]:
cond = (df["destination_port_lon"].isNotNull()) & (df["destination_port_lon"] != "-")
df.select("destination_port_lon", "destination_port_lon").filter(cond).show(5)

+--------------------+--------------------+
|destination_port_lon|destination_port_lon|
+--------------------+--------------------+
|            26.9501E|            26.9501E|
|            89.5883E|            89.5883E|
|            72.6159E|            72.6159E|
|            32.3886E|            32.3886E|
|            32.3886E|            32.3886E|
+--------------------+--------------------+
only showing top 5 rows



In [7]:
def clean_directional_column(df: DataFrame, col_name: str) -> DataFrame:
    """
    Clean a directional coordinate column and convert it to a signed double.
    Overwrites the original column.

    Args:
        df: Input Spark DataFrame
        col_name: Column to clean (e.g., 'destination_port_lon' or 'destination_port_lat')

    Returns:
        DataFrame with overwritten column as double
    """
    
    c = col(col_name)
    cleaned = when(
        c.endswith("N") | c.endswith("E"),
        regexp_replace(c, "[NE]", "").cast("double")
    ).when(
        c.endswith("S") | c.endswith("W"),
        -1 * regexp_replace(c, "[SW]", "").cast("double")
    ).otherwise(None)

    return df.withColumn(col_name, cleaned)


df = clean_directional_column(df, "destination_port_lat")
df = clean_directional_column(df, "destination_port_lon")

df.select("destination_port_lon", "destination_port_lon").filter(cond).show(50)

+--------------------+--------------------+
|destination_port_lon|destination_port_lon|
+--------------------+--------------------+
|             26.9501|             26.9501|
|             89.5883|             89.5883|
|             72.6159|             72.6159|
|             32.3886|             32.3886|
|             32.3886|             32.3886|
|             89.5883|             89.5883|
|             72.6159|             72.6159|
|             44.9496|             44.9496|
|             89.5883|             89.5883|
|             72.6159|             72.6159|
|             29.7132|             29.7132|
|             44.9496|             44.9496|
|             29.8059|             29.8059|
|             33.2242|             33.2242|
|             -0.9819|             -0.9819|
|              39.116|              39.116|
|                91.8|                91.8|
|             31.7534|             31.7534|
+--------------------+--------------------+



In [8]:
# port_locations = {
#     "Aliaga": (38.7994, 26.9366),  # Turkey
#     "EGACT": (31.2653, 32.3019),   # Port Said Container Terminal, Egypt
#     "Hazira": (21.1167, 72.6167),  # India
#     "Wadi Feiran": (28.7167, 33.6167),  # Egypt (Sinai)
#     "RAS SADAT": (30.1000, 32.3167),  # Approx, Egypt
#     "ZET_BAY EG": (31.2667, 32.3000),  # Likely Port Said, Egypt
#     "Jeddah": (21.4858, 39.1925),  # Saudi Arabia
#     "El Dekheila": (31.1333, 29.8333),  # Alexandria, Egypt
#     "RAS SHUKIER": (27.8333, 33.6167),  # Red Sea, Egypt
#     "Chittagong": (22.3331, 91.8346),  # Bangladesh
#     "AIN SUKHNA/EGY": (29.6000, 32.3167),  # Ain Sokhna, Egypt
#     "RAS SADAT EG": (30.1000, 32.3167),  # Egypt
#     "Ain Sukhna Oil Field Term.": (29.6000, 32.3167),  # Same area
#     "PORT SAID/EGY": (31.2653, 32.3019),  # Port Said, Egypt
#     "EGPSE": (31.2653, 32.3019),  # Port Said East, Egypt (approx)
#     "RAS SADAT_EG": (30.1000, 32.3167),  # Egypt
#     "Aden": (12.7856, 45.0187),  # Yemen
#     "RAS SHUKHEIR/EGY": (27.8333, 33.6167),  # Egypt
#     "Mongla": (22.4883, 89.5840),  # Bangladesh
#     "EGSAD": (30.1000, 32.3167),  # Sadat, Egypt (approx guess)
#     "JEDDAH<>SAWAKIN": (20.1172, 37.3289),  # Midpoint of route
#     "RAS GHARIB": (28.3500, 33.1000),  # Egypt
#     "SUEZ-SAFAGA": (26.7333, 33.9333),  # Near Safaga, Egypt
#     "RAS GHARIB\\EGY": (28.3500, 33.1000),  # Egypt
#     "RAS SADAT/EGY": (30.1000, 32.3167),  # Egypt
#     "WADI FEIRAN EG": (28.7167, 33.6167),  # Egypt
#     "NEOM": (28.2170, 34.6421),  # Saudi Arabia (NEOM project)
#     "SAFAGA EGY": (26.7500, 33.9333),  # Egypt
#     "Dumyat (Damietta)": (31.4167, 31.8133),  # Egypt
#     "Yarimca": (40.7619, 29.8194),  # Turkey
#     "ALEXADRIA/EGY": (31.2001, 29.9187),  # Alexandria, Egypt
#     "CNMCH": (22.9833, 113.3667),  # Possibly Machong Port, China (?)
#     "Cartagena": (10.3910, -75.4794),  # Colombia
#     "Destination not available": (None, None),
#     "-": (None, None),
#     None: (None, None)
# }

# from pyspark.sql import Row

# # Convert to Row list
# port_rows = [Row(destination_port_name=k, lat=v[0], lon=v[1]) for k, v in port_locations.items()]

# # Create DataFrame
# ports_df = spark.createDataFrame(port_rows)

# # Join
# df = df.join(ports_df, on="destination_port_name", how="left")
# cond = (df["destination_port_lon"].isNotNull()) & (df["destination_port_lon"] != "-")
# x = df.select("destination_port_lon", "destination_port_lon").filter(cond).count()


In [9]:
def clean_and_convert_date_column(
    df: DataFrame,
    column_name: str,
    year: str = "2025"
) -> DataFrame:
    """
    Clean a date string column (e.g., 'Jul 13, 17:30 UTC') and convert it to a date (yyyy-MM-dd).
    Appends the given year if missing.

    Args:
        df: Input Spark DataFrame
        column_name: Name of the date string column (e.g., 'departure_date')
        year: Year to append for parsing (default = '2025')

    Returns:
        DataFrame with the same column name converted to date (DateType)
    """
    # Clean: remove "UTC" and trim
    cleaned_col = trim(regexp_replace(column_name, "UTC", ""))

    # Rebuild full string: "Jul 13, 17:30" => "Jul 13 2025"
    full_date_str = concat_ws(" ", cleaned_col.substr(1, 6), lit(year))

    # Convert to DateType: "MMM d yyyy"
    df = df.withColumn(column_name, to_date(full_date_str, "MMM d yyyy"))

    return df

df = clean_and_convert_date_column(df, "departure_date", year="2025")
df = clean_and_convert_date_column(df, "arrival_date", "2025")

In [10]:
def fill_missing_dates_with_max(df):
    """
    Fill null values in departure_date and arrival_date columns
    with the maximum of departure_date, arrival_date, and report_date.
    """
    G = df.replace('-', None, subset=["departure_date", "arrival_date", "ingestion_date", "report_date"])
    G = G.withColumn(
        "ingestion_date",
         when(
            col("ingestion_date").isNull(),
            current_date()
        ).otherwise(col("ingestion_date"))
    ).withColumn(
        "report_date",
         when(
            col("report_date").isNull(),
            current_date()
        ).otherwise(col("report_date"))
    )
    
    df_filled = G.withColumn(
        "departure_date",
        when(
            col("departure_date").isNull(),
            least(col("departure_date"), col("arrival_date"), col("report_date"), col("ingestion_date"))
        ).otherwise(col("departure_date"))
    ).withColumn(
        "arrival_date",
        when(
            col("arrival_date").isNull(),
            greatest(col("departure_date"), col("arrival_date"), col("report_date"), col("ingestion_date"))
        ).otherwise(col("arrival_date"))
    )

    return df_filled

df = fill_missing_dates_with_max(df)

In [11]:
# df.select("departure_date", "report_date", "arrival_date", "ingestion_date").show(truncate=False)

In [24]:
# df.filter(cond).show(truncate=False)



""


In [27]:

spark.sql("CREATE DATABASE IF NOT EXISTS silver")

spark.sql("use silver")
df.write \
  .mode("append") \
  .partitionBy("ingestion_date") \
  .saveAsTable("vessels")
# df.show(2)

In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType

# Initialize Spark session
spark = SparkSession.builder.appName("PortNameMapping").getOrCreate()


# Standardized mapping for Spark
port_name_mapping = {
    # Alexandria
    "Alexandria, Egypt": "Al Iskandariyh (Alexandria)",
    "Alexandria Anch., Egypt": "Al Iskandariyh (Alexandria)",
    "ALEXADRIA/EGY": "Al Iskandariyh (Alexandria)",

    # Port Said
    "Port Said, Egypt": "Bur Sa'id",
    "Port Said Anch., Egypt": "Bur Sa'id",
    "EGACT": "Bur Sa'id",

    # Ain Sukhna
    "Ain Sukhna Oil Field Term., Egypt": "Ain Sukhna Terminal",
    "AIN SOUKHNA": "Ain Sukhna Terminal",
    "North Ain Sukhna Port": "Ain Sukhna Terminal",

    # Ismailia
    "Ismailia Anch., Egypt": "El Ismailiya",

    # As Suways / Suez
    "As Suways (Suez) Anch., Egypt": "As Suways",
    "As Suways / Suez Port, Egypt": "As Suways",
    "SUEZ_EGY": "As Suways",
    "SUEZ-SAFAGA": "As Suways",

    # Ras Shukhier
    "RAS SHUKHEIR/EGY": "Ras Shukhier",

    # Ras Gharib
    "RAS GHARIB": "Ras Gharib",

    # Ras Sadat
    "RAS SADAT_EG": "El-Adabiya",  # Closest possible match in official WPI names

    # Damietta
    "Dumyat (Damietta), Egypt": "Damietta",

    # Safaga
    "Safaga, Egypt": "Safaja",

    # El Hamra
    "El Hamra Terminal": "El Hamra Oil Terminal",
    "Jeddah, Saudi Arabia": "Jiddah",
    "Duuba, Saudi Arabia": "Duba",
    "Duba Bulk Plant Tanker Terminal": "Duba",  # merge terminal name with general port

    # Approximate / known aliases
    "JEDDAH<>SAWAKIN": "Jiddah",  # only Saudi Arabian side matters
    "Ras Al Mishab": "Ras Al Mishab",
    "Ju Aymah Oil Terminal": "Ju Aymah Oil Terminal",
    "Ju Aymah Lpg Terminal": "Ju Aymah Lpg Terminal",
    "Ras Al Ghar": "Ras Al Ghar",
    "Ras Al Khafji": "Ras Al Khafji",
    "Ras  Tannurah": "Ras  Tannurah",
    "Yanbu": "Yanbu",
    "Rabigh": "Rabigh",
    "Jizan": "Jizan",
    "King Fahd Port": "King Fahd Port",
    "Dammam": "Dammam",
    "Al Jubayl": "Al Jubayl",
    "Al Khums Anch., Libya": "Al Khair Oil Terminal",  # Assuming potential mislabeling
    "Digna (Sawakin), Sudan": "Sawakin Harbor",
    "Port Sudan, Sudan": "Port Sudan",
    "Port Sudan": "Port Sudan",
    "SAWAKIN": "Sawakin Harbor",
    "JEDDAH<>SAWAKIN": "Sawakin Harbor",
    "BEHASHER": "Beshayer Oil Terminal",  # If such typo exists in your data
    "Al Fujayrah Anch., United Arab Emirates (UAE)": "Al Fujayrah",
    "Fujairah Anch., UAE": "Al Fujayrah",
    "Fujairah": "Al Fujayrah",
    "Fujairah Anchorage": "Al Fujayrah",
    "Al Fujayrah": "Al Fujayrah",
    "Jebel Ali": "Mina Jabal Ali",
    "Jebel Ali, UAE": "Mina Jabal Ali",
    "Mina Jebel Ali": "Mina Jabal Ali",
    "Port of Jebel Ali": "Mina Jabal Ali",
    "Ruways": "Jabal Az Zannah/ruways",
    "Ruwais": "Jabal Az Zannah/ruways",
    "Das Island": "Jazirat Das",
    "Umm al-Quwain": "Umm Al Qaywayn",
    "Sharjah": "Ash Shariqah",
    "Sharjah Offshore": "Sharjah Offshore Terminal",
    "Dubai": "Dubayy",
    "Abu Dhabi": "Abu Zaby",
    "Abu Zaby": "Abu Zaby",
    "Ajman, UAE": "Ajman",
    "Umm Al Nar": "Umm An Nar",
    "Las Palmas, Spain": "Las Palmas", 
    "Ravenna Anch., Italy": "Ravenna", 
    
    "La Pallice, France": "La Pallice", 
    "Beirut Anch., Lebanon": "Bayrut",
    "Beirut": "Bayrut",
    "Beyrouth": "Bayrut",
    "Bayrut": "Bayrut",

    "Tarabulus": "Tarabulus",
    "Tripoli": "Tarabulus",  # Common English name

    "Sayda": "Sayda",
    "Saida": "Sayda",
    "Sidon": "Sayda",

    "Sidon/zahrani Terminal": "Sidon/zahrani Terminal",
    "Zahrani": "Sidon/zahrani Terminal",

    "Selaata": "Selaata",
    "Abu Khammash": "Abu Khammash",

    "Darnah": "Darnah",
    "Derna": "Darnah",

    "Az Zawiya": "Az Zawiya",
    "Zawiya": "Az Zawiya",

    "Marsa Sabratah": "Marsa Sabratah",
    "Sabrata": "Marsa Sabratah",

    "Banghazi": "Banghazi",
    "Bengasi": "Banghazi",
    "Benghazi": "Banghazi",

    "Bouri Oil Field": "Bouri Oil Field",

    "Mersa Tobruq": "Mersa Tobruq",
    "Marsa Al Hariga": "Mersa Tobruq",
    "Marsa Tobruk": "Mersa Tobruq",
    "Tobruk": "Mersa Tobruq",

    "Mina Tarabulus (Tripoli)": "Mina Tarabulus (Tripoli)",
    "Tripoli": "Mina Tarabulus (Tripoli)",
    "Tripoli Anch., Libya": "Mina Tarabulus (Tripoli)",

    "As Sidr": "As Sidr",
    "Abyar As Sirs": "As Sidr",
    "Es Sider": "As Sidr",

    "Al Burayqah": "Al Burayqah",
    "Marsa Brega": "Al Burayqah",

    "Ez Zueitina": "Ez Zueitina",
    "Qarnat Az Zuwaytinan": "Ez Zueitina",
    "Sidi Alib": "Ez Zueitina",

    "Ras   Lanuf": "Ras Lanuf",
    "Ras Lanuf": "Ras Lanuf",

    "Misratah": "Misratah",
    "Marsa Misratah": "Misratah",
    "Misurata Marina": "Misratah",
    "Qasr Ahmed": "Misratah",

    "Khoms": "Khoms",
    "Al Khums Anch., Libya": "Khoms",
    
    "Keppel - (East Singapore)": "Keppel - (East Singapore)",
    "Keppel Harbor": "Keppel - (East Singapore)",

    "Serangoon Harbor": "Serangoon Harbor",
    "Selarang": "Serangoon Harbor",

    "Jurong Island": "Jurong Island",
    "Jurong": "Jurong Island",

    "Pulau Sebarok": "Pulau Sebarok",
    "Sebarok": "Pulau Sebarok",

    "Pulau Bukom": "Pulau Bukom",
    "Bukom": "Pulau Bukom",

    "Singapore Anch. 4, Singapore": "Jurong Island",
    "Singapore Anch., Singapore": "Jurong Island",
    "Singapore": "Jurong Island",
    "Chittagong": "Chittagong",
    "Chattogram": "Chittagong",
    "Chattogram Anch.": "Chittagong",
    "Chattogram Anch., Bangladesh": "Chittagong",

    "Mongla": "Mongla",
    "Chalna Bazar": "Mongla",
    

    "Destination not available": None,
    "IN-ORDER": None,
    "": None
}


# Convert to list of tuples
mapping_data = [(k, v) for k, v in port_name_mapping.items()]

# Define schema
schema = StructType([
    StructField("raw_port_name", StringType(), True),
    StructField("standard_port_name", StringType(), True)
])

# Create DataFrame
mapping_df = spark.createDataFrame(mapping_data, schema=schema)

# Create or replace temporary view
mapping_df.createOrReplaceTempView("port_name_mapping_view")


""


In [ ]:
df.createOrReplaceTempView("vessels_temp")


In [166]:
# spark.sql("select * from vessels_temp where destination_port_lat is not null")

id,name,url,type,year_built,gross_tonnage,deadweight,length_m,beam_m,detail_link,departure_date,last_port_country,last_port_name,arrival_date,destination_port_country,destination_port_name,destination_port_lat,destination_port_lon,reported_status,report_date,ingestion_date
84,CAPT.KATTELMANN,/vessels/details/...,Cargo ship,2006,15995,20305,170.0,25.0,null,2025-07-13,Egypt,Dumyat (Damietta),2025-07-12,Turkey,Aliaga,38.8435,26.9501,Under way,2025-07-13,2025-07-24
81,WADI FERAN,/vessels/details/...,Cargo ship,2011,33234,57282,190.0,32.0,null,2025-07-16,Bangladesh,Chattogram Anch.,2025-07-16,Bangladesh,Mongla,22.4998,89.5883,-,2025-07-21,2025-07-24
82,WADI ALBOSTAN,/vessels/details/...,Cargo ship,2011,33234,57320,190.0,32.0,null,2025-07-16,Singapore,Singapore Anch. 4,2025-07-16,India,Hazira,21.0666,72.6159,-,2025-07-21,2025-07-24
0,ATHINEA,/vessels/details/...,Tanker,2006,60007,107160,248.0,43.0,null,2025-07-12,Egypt,As Suways (Suez) ...,2025-07-19,Egypt,Ain Sukhna Oil Fi...,29.5799,32.3886,-,2025-07-19,2025-07-24
1,ETC RAMSIS,/vessels/details/...,Tanker,2003,59574,110673,246.0,42.0,null,2025-07-17,Egypt,As Suways (Suez) ...,2025-07-19,Egypt,Ain Sukhna Oil Fi...,29.5799,32.3886,-,2025-07-19,2025-07-24
81,WADI FERAN,/vessels/details/...,Cargo ship,2011,33234,57282,190.0,32.0,null,2025-07-16,Bangladesh,Chattogram Anch.,2025-07-16,Bangladesh,Mongla,22.4998,89.5883,-,2025-07-19,2025-07-24
82,WADI ALBOSTAN,/vessels/details/...,Cargo ship,2011,33234,57320,190.0,32.0,null,2025-07-16,Singapore,Singapore Anch. 4,2025-07-16,India,Hazira,21.0666,72.6159,-,2025-07-19,2025-07-24
89,ELREEDY STAR,/vessels/details/...,Cargo ship,1998,13066,20537,153.0,24.0,null,2025-07-13,Djibouti,Djibouti,2025-07-14,Yemen,Aden,12.7735,44.9496,-,2025-07-19,2025-07-24
81,WADI FERAN,/vessels/details/...,Cargo ship,2011,33234,57282,190.0,32.0,null,2025-07-16,Bangladesh,Chattogram Anch.,2025-07-16,Bangladesh,Mongla,22.4998,89.5883,-,2025-07-17,2025-07-24
82,WADI ALBOSTAN,/vessels/details/...,Cargo ship,2011,33234,57320,190.0,32.0,null,2025-07-16,Singapore,Singapore Anch. 4,2025-07-16,India,Hazira,21.0666,72.6159,-,2025-07-17,2025-07-24


In [171]:
# spark.sql("""
#     SELECT vessels_temp.*,  port_name_mapping_view.standard_port_name
#     FROM vessels_temp 
#     LEFT JOIN port_name_mapping_view ON raw_port_name = destination_port_name
#     where port_name_mapping_view.standard_port_name is not null
# """)

id,name,url,type,year_built,gross_tonnage,deadweight,length_m,beam_m,detail_link,departure_date,last_port_country,last_port_name,arrival_date,destination_port_country,destination_port_name,destination_port_lat,destination_port_lon,reported_status,report_date,ingestion_date,standard_port_name
4,AL NABILA 6,/vessels/details/...,Tanker (HAZ-A),2003,22184,34810,171.0,27.0,null,2025-07-17,Egypt,Alexandria,2025-07-11,null,ALEXADRIA/EGY,null,null,Under way,2025-07-17,2025-07-24,Al Iskandariyh (A...
4,AL NABILA 6,/vessels/details/...,Tanker (HAZ-A),2003,22184,34810,171.0,27.0,null,2025-07-11,Egypt,Alexandria Anch.,2025-07-11,null,ALEXADRIA/EGY,null,null,At anchor,2025-07-12,2025-07-24,Al Iskandariyh (A...
4,AL NABILA 6,/vessels/details/...,Tanker (HAZ-A),2003,22184,34810,171.0,27.0,null,2025-07-11,Egypt,Alexandria Anch.,2025-07-11,null,ALEXADRIA/EGY,null,null,-,2025-07-13,2025-07-24,Al Iskandariyh (A...
91,EGY SKY,/vessels/details/...,Cargo ship (HAZ-A),2007,10308,12720,149.0,23.0,null,2025-07-11,Egypt,Alexandria Anch.,2025-07-12,null,EGACT,null,null,At anchor,2025-07-12,2025-07-24,Bur Sa'id
88,AL HURREYA,/vessels/details/...,Cargo ship,2005,13529,5904,140.0,24.0,null,2025-07-07,Egypt,Safaga,2025-07-24,null,SUEZ-SAFAGA,null,null,Under way,2025-07-07,2025-07-24,As Suways
87,AL HURREYA,/vessels/details/...,Cargo ship,2005,13529,5904,140.0,24.0,null,2025-07-17,Egypt,Safaga,2025-07-24,null,SUEZ-SAFAGA,null,null,Moored,2025-07-17,2025-07-24,As Suways
87,AL HURREYA,/vessels/details/...,Cargo ship,2005,13529,5904,140.0,24.0,null,2025-07-19,Egypt,Safaga,2025-07-24,null,SUEZ-SAFAGA,null,null,Moored,2025-07-19,2025-07-24,As Suways
87,AL HURREYA,/vessels/details/...,Cargo ship,2005,13529,5904,140.0,24.0,null,2025-07-21,Egypt,Safaga,2025-07-24,null,SUEZ-SAFAGA,null,null,-,2025-07-21,2025-07-24,As Suways
87,AL HURREYA,/vessels/details/...,Cargo ship,2005,13529,5904,140.0,24.0,null,2025-07-12,Egypt,Safaga,2025-07-24,null,SUEZ-SAFAGA,null,null,Moored,2025-07-12,2025-07-24,As Suways
87,AL HURREYA,/vessels/details/...,Cargo ship,2005,13529,5904,140.0,24.0,null,2025-07-13,Egypt,Safaga,2025-07-24,null,SUEZ-SAFAGA,null,null,Moored,2025-07-13,2025-07-24,As Suways


In [148]:
  # spark.stop()